# **SETUP & LIBRARIES**

In [1]:
import pandas as pd
import os
import sys

# allow src imports
sys.path.append("../..")

from src.data_loader import load_dataset
from src.feature_engineering import convert_time, compute_time_diff
from src.metrics import summarize_times, format_time

# load germany dataset
data = load_dataset("germany")

stops = data["stops"]
stop_times = data["stop_times"]
trips = data["trips"]
routes = data["routes"]

stops.head(10)

,stop_name,parent_station,stop_id,stop_lat,stop_lon,location_type,platform_code
0,'s-Heerenberg Gouden Handen,NaN,634349,51.872250,6.247338,1.0,NaN
1,'s-Heerenberg Gouden Handen,634349.0,525735,51.872280,6.247406,NaN,NaN
2,'s-Heerenberg Molenpoort,NaN,145838,51.876320,6.247432,1.0,NaN
3,'s-Heerenberg Molenpoort,145838.0,16932,51.876320,6.247432,NaN,NaN
4,'s-Heerenberg Muziekschool,NaN,500876,51.874790,6.254206,1.0,NaN
5,'s-Heerenberg Muziekschool,500876.0,445895,51.874844,6.254296,NaN,NaN
6,'s-Heerenberg Muziekschool,500876.0,681268,51.874737,6.254116,NaN,NaN
7,'s-Hertogenbosch,NaN,353412,51.690540,5.293723,1.0,NaN
8,'s-Hertogenbosch,353412.0,134900,51.690540,5.293723,NaN,NaN
9,"(Planung) Brokdorf, Peuser (West), Mast 2r",487040.0,547672,53.869690,9.342641,NaN,NaN


# **BASIC DATA OVERVIEW**

In [2]:
# inspect stops
print("\nStop Columns:")
for col in stops.columns:
    print(f"- {col}")


Stop Columns:
- stop_name
- parent_station
- stop_id
- stop_lat
- stop_lon
- location_type
- platform_code


In [3]:
# inspect stop_times
print("\nStop Times Columns:")
for col in stop_times.columns:
    print(f"- {col}")


Stop Times Columns:
- trip_id
- arrival_time
- departure_time
- stop_id
- stop_sequence
- pickup_type
- drop_off_type


In [4]:
print("Total Stops:", stops.shape[0])
print("Total Stop Times:", stop_times.shape[0])
print("Total Trips:", trips.shape[0])
print("Total Routes:", routes.shape[0])

Total Stops: 683842
Total Stop Times: 34174652
Total Trips: 1656141
Total Routes: 24744


# **FEATURE ENGINEERING**

In [5]:
# convert time to timedelta
stop_times = convert_time(stop_times)

# compute inter-stop time difference
stop_times = compute_time_diff(stop_times)

# create previous stop reference BEFORE merges
stop_times["prev_stop_id"] = stop_times.groupby("trip_id")["stop_id"].shift(1)

# light filtering only (do not over-filter)
stop_times = stop_times[
    (stop_times["time_diff"] > pd.Timedelta(seconds=20)) &
    (stop_times["time_diff"] <= pd.Timedelta(hours=2))
].copy()

stop_times.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,arrival_time_td,time_diff,prev_stop_id
1355286,1,13:40:00,13:40:00,74706,1,NaN,NaN,0 days 13:40:00,0 days 00:01:00,413209.0
1355287,1,13:42:00,13:42:00,291577,2,NaN,NaN,0 days 13:42:00,0 days 00:02:00,74706.0
1355288,1,13:43:00,13:43:00,244762,3,NaN,NaN,0 days 13:43:00,0 days 00:01:00,291577.0
1355289,1,13:45:00,13:45:00,197159,4,NaN,NaN,0 days 13:45:00,0 days 00:02:00,244762.0
1355290,1,13:46:00,13:46:00,337171,5,NaN,NaN,0 days 13:46:00,0 days 00:01:00,197159.0


# **SYSTEM-WIDE TRAVEL TIME INSIGHTS**

In [6]:
# summary stats
summary = summarize_times(stop_times["time_diff"])
print(summary)

{'count': 30835401, 'mean': Timedelta('0 days 00:01:51.883144344'), 'min': Timedelta('0 days 00:00:21'), '25%': Timedelta('0 days 00:01:00'), '50%': Timedelta('0 days 00:01:00'), '75%': Timedelta('0 days 00:02:00'), 'max': Timedelta('0 days 02:00:00')}


# **DATA MERGING**

In [7]:
# merge trips
final_df = stop_times.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

# merge routes
final_df = final_df.merge(
    routes[["route_id", "route_short_name", "route_type"]],
    on="route_id",
    how="left"
)

# merge stop names
final_df = final_df.merge(
    stops[["stop_id", "stop_name"]],
    on="stop_id",
    how="left"
)

# merge previous stop names
final_df = final_df.merge(
    stops[["stop_id", "stop_name"]].rename(
        columns={"stop_id": "prev_stop_id", "stop_name": "prev_stop_name"}
    ),
    on="prev_stop_id",
    how="left"
)

final_df.head(10)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,arrival_time_td,time_diff,prev_stop_id,route_id,route_short_name,route_type,stop_name,prev_stop_name
0,1,13:40:00,13:40:00,74706,1,NaN,NaN,0 days 13:40:00,0 days 00:01:00,413209.0,18268,6,0,Alsbach Beuneweg Straßenbahn,Alsbach Melibokusschule Bus NHähnlein
1,1,13:42:00,13:42:00,291577,2,NaN,NaN,0 days 13:42:00,0 days 00:02:00,74706.0,18268,6,0,Jugenheim Bickenbacher Straße Haltestelle 0,Alsbach Beuneweg Straßenbahn
2,1,13:43:00,13:43:00,244762,3,NaN,NaN,0 days 13:43:00,0 days 00:01:00,291577.0,18268,6,0,Jugenheim Ludwigstraße Haltestelle 0,Jugenheim Bickenbacher Straße Haltestelle 0
3,1,13:45:00,13:45:00,197159,4,NaN,NaN,0 days 13:45:00,0 days 00:02:00,244762.0,18268,6,0,Seeheim Tannenbergstraße Haltestelle 0,Jugenheim Ludwigstraße Haltestelle 0
4,1,13:46:00,13:46:00,337171,5,NaN,NaN,0 days 13:46:00,0 days 00:01:00,197159.0,18268,6,0,Seeheim Neues Rathaus Haltestelle 0,Seeheim Tannenbergstraße Haltestelle 0
5,1,13:48:00,13:48:00,585767,6,NaN,NaN,0 days 13:48:00,0 days 00:02:00,337171.0,18268,6,0,Seeheim Im Güldenen Wingert Haltestelle 0,Seeheim Neues Rathaus Haltestelle 0
6,1,13:51:00,13:51:00,47294,7,NaN,NaN,0 days 13:51:00,0 days 00:03:00,585767.0,18268,6,0,Malchen Haltestelle 0,Seeheim Im Güldenen Wingert Haltestelle 0
7,1,13:53:00,13:53:00,486333,8,NaN,NaN,0 days 13:53:00,0 days 00:02:00,47294.0,18268,6,0,Eberstadt Mittelschneise Haltestelle 0,Malchen Haltestelle 0
8,1,13:55:00,13:55:00,209435,9,NaN,NaN,0 days 13:55:00,0 days 00:02:00,486333.0,18268,6,0,Eberstadt Frankenstein Haltestelle 0,Eberstadt Mittelschneise Haltestelle 0
9,1,13:56:00,13:56:00,201937,10,NaN,NaN,0 days 13:56:00,0 days 00:01:00,209435.0,18268,6,0,Eberstadt Friedhof Haltestelle 0,Eberstadt Frankenstein Haltestelle 0


# **TRANSPORT MODE MAPPING**

In [8]:
# map transport types
route_type_map = {
    0: "Tram",
    1: "Subway",
    2: "Rail",
    3: "Bus",
    100: "Rail",
    109: "Suburban Rail",
    400: "Subway",
    700: "Bus"
}

final_df["route_type_name"] = final_df["route_type"].map(route_type_map).fillna("Other")

print("Unknown route types:", (final_df["route_type_name"] == "Other").sum())

Unknown route types: 35433


# **RAIL CLASSIFICATION**

In [9]:
def classify_rail(row):
    route = str(row["route_short_name"]).upper().strip()

    if row["route_type_name"] == "Rail":

        # S-Bahn detection (must contain digit)
        if route.startswith("S") and any(char.isdigit() for char in route):
            return "S-Bahn"

        elif route.startswith(("RE", "RB")):
            return "Regional"

        elif route.startswith(("ICE", "IC", "EC")):
            return "Long-Distance"

        else:
            return "Other Rail"

    return None

final_df["rail_type"] = final_df.apply(classify_rail, axis=1)

# **REGIONAL DATASET CREATION (CORE OUTPUT)**

In [10]:
# keep only intercity rail
regional_df = final_df[
    final_df["rail_type"].isin(["Regional", "Long-Distance"])
].copy()

# realistic intercity travel times
regional_df = regional_df[
    (regional_df["time_diff"] >= pd.Timedelta(minutes=3)) &
    (regional_df["time_diff"] <= pd.Timedelta(hours=3))
]

regional_df.head(10)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,arrival_time_td,time_diff,prev_stop_id,route_id,route_short_name,route_type,stop_name,prev_stop_name,route_type_name,rail_type
614,38,23:13:00,23:14:00,610039,1,NaN,NaN,0 days 23:13:00,0 days 00:03:00,282241.0,18851,RB51,2,Regensburg-Burgweinting,Regensburg Hbf,Rail,Regional
615,38,23:16:00,23:17:00,158809,2,NaN,NaN,0 days 23:16:00,0 days 00:03:00,610039.0,18851,RB51,2,Obertraubling,Regensburg-Burgweinting,Rail,Regional
616,38,23:25:00,23:25:00,416684,3,NaN,NaN,0 days 23:25:00,0 days 00:09:00,158809.0,18851,RB51,2,Sünching,Obertraubling,Rail,Regional
617,38,23:29:00,23:30:00,222713,4,NaN,NaN,0 days 23:29:00,0 days 00:04:00,416684.0,18851,RB51,2,Radldorf(Niederbay),Sünching,Rail,Regional
618,38,23:36:00,23:38:00,530708,5,NaN,NaN,0 days 23:36:00,0 days 00:07:00,222713.0,18851,RB51,2,Straubing,Radldorf(Niederbay),Rail,Regional
619,38,23:43:00,23:44:00,102073,6,NaN,NaN,0 days 23:43:00,0 days 00:07:00,530708.0,18851,RB51,2,Straßkirchen,Straubing,Rail,Regional
620,38,23:52:00,23:52:00,330332,7,NaN,NaN,0 days 23:52:00,0 days 00:09:00,102073.0,18851,RB51,2,Plattling,Straßkirchen,Rail,Regional
1704,94,17:47:00,17:48:00,51078,1,NaN,NaN,0 days 17:47:00,0 days 00:03:00,49182.0,2628,RE6,2,Kiebingen,Rottenburg(Neckar),Rail,Regional
1705,94,17:54:00,17:54:00,83699,2,NaN,NaN,0 days 17:54:00,0 days 00:07:00,51078.0,2628,RE6,2,Tübingen Hbf,Kiebingen,Rail,Regional
2316,121,08:42:00,08:44:00,116212,1,NaN,NaN,0 days 08:42:00,0 days 00:05:00,571924.0,8498,RB58,2,Kreuzstraße,Holzkirchen,Rail,Regional


# **AGGREGATION (SEGMENT LEVEL)**

In [11]:
regional_dashboard = regional_df.groupby(
    ["route_short_name", "prev_stop_name", "stop_name"]
).agg(
    avg_travel_time=("time_diff", "mean"),
    trip_count=("time_diff", "count")
).reset_index()

# readable format
regional_dashboard["avg_time_readable"] = regional_dashboard["avg_travel_time"].apply(format_time)

regional_dashboard.head(10)

,route_short_name,prev_stop_name,stop_name,avg_travel_time,trip_count,avg_time_readable
0,EC,Basel SBB,Zürich HB,0 days 01:27:00,2,87 Min 0 Sec
1,EC,Bregenz,Lindau-Reutin,0 days 00:08:21.176470588,17,8 Min 21 Sec
2,EC,Bregenz,St. Margrethen SG,0 days 00:11:00,16,11 Min 0 Sec
3,EC,Decin hl.n.,Usti nad Labem hl.n.,0 days 00:14:00,2,14 Min 0 Sec
4,EC,Dobova,Dobova(Gr),0 days 00:23:00,1,23 Min 0 Sec
5,EC,Dobova(Gr),Zapresic,0 days 00:10:00,1,10 Min 0 Sec
6,EC,Jesenice(Gr),Jesenice(SLO),0 days 00:10:00,1,10 Min 0 Sec
7,EC,Jesenice(SLO),Lesce-Bled,0 days 00:16:00,1,16 Min 0 Sec
8,EC,Kranj,Ljubljana,0 days 00:28:00,1,28 Min 0 Sec
9,EC,Lesce-Bled,Kranj,0 days 00:26:00,1,26 Min 0 Sec


# **EXPORT**

In [12]:
regional_dashboard.to_csv("../../data/processed/germany/regional_dashboard.csv", index=False)

print("Germany regional dataset saved")

Germany regional dataset saved


# # **FINAL VALIDATION CHECKS**

In [13]:
print("\nAll Modes:")
print(final_df["route_type_name"].value_counts())

print("\nRail Types:")
print(final_df["rail_type"].value_counts())

print("\nRegional Breakdown:")
print(regional_df["rail_type"].value_counts())


All Modes:
route_type_name
Bus       25469521
Tram       2886318
Rail       1304056
Subway     1140073
Other        35433
Name: count, dtype: int64

Rail Types:
rail_type
S-Bahn           692850
Regional         466111
Other Rail        99788
Long-Distance     45307
Name: count, dtype: int64

Regional Breakdown:
rail_type
Regional         432772
Long-Distance     45130
Name: count, dtype: int64
